# 04 Matchup Features

Combine batter weakness and pitcher pitch mix into matchup scores.



In [2]:
import sys
from pathlib import Path

import pandas as pd
import numpy as np

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

from src.data_loader import save_csv

print(PROJECT_ROOT)

/Users/henry_tsai/Desktop/daily-mlb-hitter-forecasting


In [3]:
pa_path = PROJECT_ROOT / "data/processed/pa_2025_04_01_to_05_31.csv"
weakness_path = PROJECT_ROOT / "data/features/batter_weakness_profiles_2025_04_01_to_05_31.csv"

pa_df = pd.read_csv(pa_path)
pa_df["game_date"] = pd.to_datetime(pa_df["game_date"])

batter_weakness = pd.read_csv(weakness_path)

print(pa_df.shape)
print(batter_weakness.shape)

pa_df.head()

(60470, 13)
(535, 30)


,game_date,batter,pitcher,events,description,pitch_type,stand,p_throws,launch_speed,launch_angle,estimated_woba_using_speedangle,woba_value,xwoba_value
0,2025-05-31,691785,606160,field_out,hit_into_play,FF,L,R,91.3,33.0,0.057000,0.0,0.057000
1,2025-05-31,665966,606160,field_out,hit_into_play,SL,R,R,71.4,-20.0,0.036000,0.0,0.036000
2,2025-05-31,677800,606160,field_out,hit_into_play,FF,L,R,103.4,36.0,0.975773,0.0,0.975773
3,2025-05-31,663897,595897,strikeout,swinging_strike,SL,R,R,NaN,NaN,0.000000,0.0,0.000000
4,2025-05-31,671739,595897,field_out,hit_into_play,SL,L,R,100.8,4.0,0.542000,0.0,0.542000


In [4]:
def map_pitch_group(pitch_type):
    if pd.isna(pitch_type):
        return "unknown"

    fastball = {"FF", "FA"}
    sinker = {"SI"}
    cutter = {"FC"}
    slider = {"SL", "ST"}
    curveball = {"CU", "KC", "SV"}
    changeup = {"CH", "FS", "FO"}

    if pitch_type in fastball:
        return "fastball"
    if pitch_type in sinker:
        return "sinker"
    if pitch_type in cutter:
        return "cutter"
    if pitch_type in slider:
        return "slider"
    if pitch_type in curveball:
        return "curveball"
    if pitch_type in changeup:
        return "changeup"

    return "other"


pa_df["pitch_group"] = pa_df["pitch_type"].apply(map_pitch_group)

pa_df["pitch_group"].value_counts()

pitch_group
fastball     18540
slider       13377
sinker       10087
changeup      9022
curveball     4618
cutter        4484
unknown        194
other          148
Name: count, dtype: int64

In [5]:
pitcher_pitch_counts = (
    pa_df
    .groupby(["pitcher", "pitch_group"])
    .size()
    .reset_index(name="pitch_count")
)

pitcher_total = (
    pitcher_pitch_counts
    .groupby("pitcher")["pitch_count"]
    .sum()
    .reset_index(name="total_pitch_count")
)

pitcher_pitch_counts = pitcher_pitch_counts.merge(
    pitcher_total,
    on="pitcher",
    how="left"
)

pitcher_pitch_counts["usage"] = (
    pitcher_pitch_counts["pitch_count"] / pitcher_pitch_counts["total_pitch_count"]
)

pitcher_pitch_counts.head()

,pitcher,pitch_group,pitch_count,total_pitch_count,usage
0,434378,changeup,15,204,0.073529
1,434378,curveball,23,204,0.112745
2,434378,fastball,98,204,0.480392
3,434378,slider,68,204,0.333333
4,445276,cutter,56,69,0.811594


In [6]:
pitcher_pitch_mix = (
    pitcher_pitch_counts
    .pivot(index="pitcher", columns="pitch_group", values="usage")
    .fillna(0)
    .add_prefix("pitcher_usage_")
    .reset_index()
)

print(pitcher_pitch_mix.shape)
pitcher_pitch_mix.head()

(641, 9)


pitch_group,pitcher,pitcher_usage_changeup,pitcher_usage_curveball,pitcher_usage_cutter,pitcher_usage_fastball,pitcher_usage_other,pitcher_usage_sinker,pitcher_usage_slider,pitcher_usage_unknown
0,434378,0.073529,0.112745,0.000000,0.480392,0.0,0.000000,0.333333,0.000000
1,445276,0.000000,0.000000,0.811594,0.000000,0.0,0.101449,0.086957,0.000000
2,445926,0.166667,0.000000,0.333333,0.000000,0.0,0.500000,0.000000,0.000000
3,450203,0.099502,0.383085,0.084577,0.268657,0.0,0.164179,0.000000,0.000000
4,455119,0.187500,0.000000,0.343750,0.296875,0.0,0.109375,0.046875,0.015625


In [7]:
batter_pitcher_pairs = (
    pa_df
    .groupby(["batter", "pitcher"])
    .agg(
        matchup_PA=("events", "count"),
        matchup_xwOBA=("xwoba_value", "mean"),
        matchup_wOBA=("woba_value", "mean"),
    )
    .reset_index()
)

print(batter_pitcher_pairs.shape)
batter_pitcher_pairs.head()

(34205, 5)


,batter,pitcher,matchup_PA,matchup_xwOBA,matchup_wOBA
0,455117,543294,2,0.0000,0.0
1,455117,592332,1,0.1470,0.0
2,455117,592791,2,0.0565,0.0
3,455117,594902,2,0.0045,0.0
4,455117,596133,1,0.2400,0.0


In [8]:
matchup_df = (
    batter_pitcher_pairs
    .merge(batter_weakness, on="batter", how="left")
    .merge(pitcher_pitch_mix, on="pitcher", how="left")
)

print(matchup_df.shape)
matchup_df.head()

(34205, 42)


,batter,pitcher,matchup_PA,matchup_xwOBA,matchup_wOBA,xwOBA_vs_changeup,xwOBA_vs_curveball,xwOBA_vs_cutter,xwOBA_vs_fastball,xwOBA_vs_other,...,main_weakness_league_xwOBA,main_weakness_PA,pitcher_usage_changeup,pitcher_usage_curveball,pitcher_usage_cutter,pitcher_usage_fastball,pitcher_usage_other,pitcher_usage_sinker,pitcher_usage_slider,pitcher_usage_unknown
0,455117,543294,2,0.0000,0.0,0.295794,0.0045,0.06475,0.339023,NaN,...,0.344215,16.0,0.394422,0.071713,0.000000,0.135458,0.0,0.390438,0.000000,0.007968
1,455117,592332,1,0.1470,0.0,0.295794,0.0045,0.06475,0.339023,NaN,...,0.344215,16.0,0.344828,0.000000,0.000000,0.581897,0.0,0.000000,0.068966,0.004310
2,455117,592791,2,0.0565,0.0,0.295794,0.0045,0.06475,0.339023,NaN,...,0.344215,16.0,0.103896,0.125541,0.134199,0.316017,0.0,0.108225,0.212121,0.000000
3,455117,594902,2,0.0045,0.0,0.295794,0.0045,0.06475,0.339023,NaN,...,0.344215,16.0,0.100000,0.056250,0.037500,0.237500,0.0,0.350000,0.218750,0.000000
4,455117,596133,1,0.2400,0.0,0.295794,0.0045,0.06475,0.339023,NaN,...,0.344215,16.0,0.300000,0.000000,0.022222,0.633333,0.0,0.000000,0.033333,0.011111


In [9]:
matchup_df.isna().mean().sort_values(ascending=False).head(30)

weakness_vs_other             0.712118
xwOBA_vs_other                0.712118
PA_vs_other                   0.712118
xwOBA_vs_unknown              0.668499
PA_vs_unknown                 0.668499
weakness_vs_unknown           0.668499
weakness_vs_curveball         0.013448
PA_vs_curveball               0.013448
xwOBA_vs_curveball            0.013448
xwOBA_vs_cutter               0.011226
weakness_vs_cutter            0.011226
PA_vs_cutter                  0.011226
main_weakness_pitch_group     0.006607
main_weakness_score           0.006607
main_weakness_xwOBA           0.006607
main_weakness_league_xwOBA    0.006607
main_weakness_PA              0.006607
PA_vs_sinker                  0.004415
weakness_vs_sinker            0.004415
xwOBA_vs_sinker               0.004415
PA_vs_changeup                0.003508
weakness_vs_changeup          0.003508
xwOBA_vs_changeup             0.003508
weakness_vs_slider            0.001666
PA_vs_slider                  0.001666
xwOBA_vs_slider          

In [10]:
pitch_groups = ["fastball", "slider", "curveball", "changeup", "cutter", "sinker"]

In [11]:
def compute_matchup_weakness_score(row, pitch_groups):
    score = 0.0

    for pitch in pitch_groups:
        weakness_col = f"weakness_vs_{pitch}"
        usage_col = f"pitcher_usage_{pitch}"

        weakness = row.get(weakness_col, np.nan)
        usage = row.get(usage_col, np.nan)

        if pd.notna(weakness) and pd.notna(usage):
            score += weakness * usage

    return score


matchup_df["matchup_weakness_score"] = matchup_df.apply(
    lambda row: compute_matchup_weakness_score(row, pitch_groups),
    axis=1
)

matchup_df[["batter", "pitcher", "matchup_PA", "matchup_xwOBA", "matchup_weakness_score"]].head()

,batter,pitcher,matchup_PA,matchup_xwOBA,matchup_weakness_score
0,455117,543294,2,0.0000,0.040450
1,455117,592332,1,0.1470,0.002978
2,455117,592791,2,0.0565,0.092852
3,455117,594902,2,0.0045,0.064052
4,455117,596133,1,0.2400,0.008405


In [12]:
matchup_df["matchup_weakness_score"].describe()

count    34205.000000
mean         0.000875
std          0.058923
min         -0.596874
25%         -0.032767
50%          0.001805
75%          0.034620
max          0.316448
Name: matchup_weakness_score, dtype: float64

In [13]:
q75 = matchup_df["matchup_weakness_score"].quantile(0.75)
q25 = matchup_df["matchup_weakness_score"].quantile(0.25)

def label_matchup(score):
    if pd.isna(score):
        return "Unknown"
    if score >= q75:
        return "Unfavorable"
    if score <= q25:
        return "Favorable"
    return "Neutral"


matchup_df["matchup_label"] = matchup_df["matchup_weakness_score"].apply(label_matchup)

matchup_df["matchup_label"].value_counts()

matchup_label
Neutral        17101
Unfavorable     8552
Favorable       8552
Name: count, dtype: int64

In [14]:
matchup_label_summary = (
    matchup_df
    .groupby("matchup_label")
    .agg(
        n_matchups=("matchup_PA", "count"),
        avg_matchup_PA=("matchup_PA", "mean"),
        actual_matchup_xwOBA=("matchup_xwOBA", "mean"),
        actual_matchup_wOBA=("matchup_wOBA", "mean"),
        avg_matchup_weakness_score=("matchup_weakness_score", "mean"),
    )
    .reset_index()
)

matchup_label_summary

,matchup_label,n_matchups,avg_matchup_PA,actual_matchup_xwOBA,actual_matchup_wOBA,avg_matchup_weakness_score
0,Favorable,8552,1.831151,0.383334,0.382887,-0.072566
1,Neutral,17101,1.801532,0.314533,0.319247,0.001545
2,Unfavorable,8552,1.637278,0.245628,0.252800,0.072976


In [15]:
matchup_df_reliable = matchup_df[matchup_df["matchup_PA"] >= 3].copy()

print(matchup_df_reliable.shape)

matchup_reliable_summary = (
    matchup_df_reliable
    .groupby("matchup_label")
    .agg(
        n_matchups=("matchup_PA", "count"),
        avg_matchup_PA=("matchup_PA", "mean"),
        actual_matchup_xwOBA=("matchup_xwOBA", "mean"),
        actual_matchup_wOBA=("matchup_wOBA", "mean"),
        avg_matchup_weakness_score=("matchup_weakness_score", "mean"),
    )
    .reset_index()
)

matchup_reliable_summary

(7750, 44)


,matchup_label,n_matchups,avg_matchup_PA,actual_matchup_xwOBA,actual_matchup_wOBA,avg_matchup_weakness_score
0,Favorable,2347,3.247550,0.385187,0.379866,-0.071625
1,Neutral,4119,3.244477,0.314388,0.317173,-0.000156
2,Unfavorable,1284,3.231308,0.250669,0.253502,0.065460


In [16]:
matchup_path = PROJECT_ROOT / "data/features/matchup_features_2025_04_01_to_05_31.csv"
pitcher_mix_path = PROJECT_ROOT / "data/features/pitcher_pitch_mix_2025_04_01_to_05_31.csv"
matchup_summary_path = PROJECT_ROOT / "reports/matchup_label_summary_2025_04_01_to_05_31.csv"

save_csv(matchup_df, matchup_path)
save_csv(pitcher_pitch_mix, pitcher_mix_path)
save_csv(matchup_label_summary, matchup_summary_path)

print(matchup_path.exists())
print(pitcher_mix_path.exists())
print(matchup_summary_path.exists())

True
True
True
